In [1]:
import os
import pandas as pd
import snowflake.connector

from dotenv import load_dotenv

load_dotenv(".env")

connection = snowflake.connector.connect(
    account=os.environ["SNOWFLAKE_ACCOUNT"],
    user=os.environ["SNOWFLAKE_USERNAME"],
    authenticator="SNOWFLAKE_JWT",
    private_key_file="rsa_key.p8",
    warehouse=os.environ["SNOWFLAKE_WAREHOUSE"],
    database=os.environ["SNOWFLAKE_DATABASE"],
    schema=os.environ["SNOWFLAKE_SCHEMA"],
    role=os.environ["SNOWFLAKE_ROLE"],
)

In [ ]:
metrics = pd.read_sql("""
    SELECT *
    FROM EXPERIMENT_COPILOT.PUBLIC.ML_MODEL_METRICS
""", connection)

drivers = pd.read_sql("""
    SELECT *
    FROM EXPERIMENT_COPILOT.PUBLIC.ML_FEATURE_IMPORTANCE
    ORDER BY IMPORTANCE DESC
    LIMIT 20
""", connection)

accounts = pd.read_sql("""
    SELECT
        ROW_NUMBER() OVER (
            ORDER BY PREDICTED_WIN_PROBABILITY_90D DESC
        ) AS PRIORITY_RANK,
        ACCOUNT_NAME,
        INDUSTRY,
        SEGMENT,
        ESTIMATED_ANNUAL_RECURRING_REVENUE AS ARR,
        PREDICTED_WIN_PROBABILITY_90D AS WIN_PROBABILITY
    FROM EXPERIMENT_COPILOT.PUBLIC.ML_ACCOUNT_SCORES
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY ACCOUNT_ID
        ORDER BY PREDICTED_WIN_PROBABILITY_90D DESC
    ) = 1
    ORDER BY WIN_PROBABILITY DESC
    LIMIT 50
""", connection)

/var/folders/35/mcj68sf96mz81d075p4nxy3m0000gp/T/ipykernel_36684/2423190565.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  metrics = pd.read_sql("""
/var/folders/35/mcj68sf96mz81d075p4nxy3m0000gp/T/ipykernel_36684/2423190565.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  drivers = pd.read_sql("""
/var/folders/35/mcj68sf96mz81d075p4nxy3m0000gp/T/ipykernel_36684/2423190565.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  accounts = pd.read_sql("""


In [3]:
def build_context():
    return f"""
MODEL METRICS
{metrics.to_string(index=False)}

IMPORTANT MODEL DRIVERS
{drivers.to_string(index=False)}

TOP-RANKED ACCOUNTS
{accounts.to_string(index=False)}
"""

In [6]:
import json
import requests

def ask_account_copilot(question):
    top_accounts = accounts.head(20).to_dict(orient="records")
    model_metrics = metrics.to_dict(orient="records")
    model_drivers = drivers.head(15).to_dict(orient="records")

    system_prompt = """
You are an Account Scoring Copilot for a sales team.

The accounts are already ranked by a machine-learning model.
A higher WIN_PROBABILITY means higher sales priority.

Rules:
1. Answer directly from the supplied data.
2. If asked which accounts to prioritize, select accounts with the
   smallest PRIORITY_RANK or highest WIN_PROBABILITY.
3. You do not need the model formula to recommend the ranked accounts.
4. Explain that each account is selected primarily because of its
   predicted win probability and priority rank.
5. Industry, segment, and ARR may provide useful business context, but
   do not claim they caused the score.
6. Global model drivers describe the overall model, not necessarily an
   individual account.
7. Never invent account-specific reasons.
8. Show probabilities as percentages.
9. Give concise, actionable answers.
"""

    data = {
        "top_ranked_accounts": top_accounts,
        "model_metrics": model_metrics,
        "global_model_drivers": model_drivers,
    }

    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": "llama3.2:3b",
            "system": system_prompt,
            "prompt": f"""
ACCOUNT SCORING DATA:
{json.dumps(data, indent=2, default=str)}

USER QUESTION:
{question}

Answer the question directly. For prioritization questions, provide a
numbered list containing account name, probability, industry, segment,
ARR, and a short sales action.
""",
            "stream": False,
            "options": {
                "temperature": 0.1,
                "num_ctx": 8192,
            },
        },
        timeout=180,
    )

    response.raise_for_status()
    return response.json()["response"]

In [7]:
print(ask_account_copilot(
    "Which five accounts should sales prioritize, and why?"
))

Based on the provided account scoring data, here are the top 5 accounts to prioritize:

1. **Town Sports International Holdings, Inc.**
	* WIN_PROBABILITY: 68.61%
	* Industry: Education
	* Segment: SMB
	* ARR: $400,000
	* Sales Action: Schedule a meeting with the decision-maker to discuss potential partnership opportunities.

2. **Deltic Timber Corporation**
	* WIN_PROBABILITY: 54.16%
	* Industry: Financial Services
	* Segment: Enterprise
	* ARR: $5,500,000
	* Sales Action: Request a call with the CEO to explore revenue growth strategies.

3. **CommScope Holding Company, Inc.**
	* WIN_PROBABILITY: 52.69%
	* Industry: Hospitality
	* Segment: SMB
	* ARR: $475,000
	* Sales Action: Schedule a meeting with the sales team to discuss potential partnership opportunities.

4. **Alliance One International, Inc.**
	* WIN_PROBABILITY: 50.53%
	* Industry: Healthcare
	* Segment: Midmarket
	* ARR: $1,300,000
	* Sales Action: Send a personalized email to the decision-maker highlighting the benefits of

In [ ]:
import rateLimit from 'express-rate-limit'

app.use(express.json({ limit: '20kb' }))

const chatLimiter = rateLimit({
  windowMs: 60_000,
  limit: 10,
  standardHeaders: true,
  legacyHeaders: false,
})


app.post('/api/account-copilot', chatLimiter, async (request, response) => {
  try {
    const question = String(request.body?.question || '').trim()
    const history = Array.isArray(request.body?.history)
      ? request.body.history.slice(-6)
      : []

    if (!question) {
      response.status(400).json({ error: 'Please enter a question.' })
      return
    }

    if (question.length > 1000) {
      response.status(400).json({ error: 'Question is too long.' })
      return
    }

    if (!process.env.GROQ_API_KEY) {
      response.status(503).json({
        error: 'Account Copilot is not configured.',
      })
      return
    }

    const dashboard = await getMlScoringDashboard()

    const context = {
      deployedModel: dashboard.deployedModel,
      models: dashboard.models,
      selectionRationale: dashboard.selectionRationale,
      calibrationSummary: dashboard.calibrationSummary,
      cohortPerformance: dashboard.cohortPerformance,
      drivers: dashboard.drivers,
      priorityAccounts: dashboard.priorityAccounts,
      accountExplanations: dashboard.accountExplanations,
    }

    const groqResponse = await fetch(
      'https://api.groq.com/openai/v1/chat/completions',
      {
        method: 'POST',
        headers: {
          Authorization: `Bearer ${process.env.GROQ_API_KEY}`,
          'Content-Type': 'application/json',
        },
        body: JSON.stringify({
          model:
            process.env.GROQ_MODEL ||
            'llama-3.3-70b-versatile',
          temperature: 0.1,
          max_completion_tokens: 900,
          messages: [
            {
              role: 'system',
              content: `
You are an Account Scoring Copilot for a sales team.

Answer only from the supplied account-scoring data.

Rules:
- Higher predicted win probability means higher sales priority.
- For prioritization questions, use priority rank and predicted probability.
- Include account names and relevant numeric evidence.
- Show probabilities as percentages.
- Industry, segment and ARR are business context, not proven causes.
- Global feature importance applies across the population, not necessarily
  to an individual account.
- Predictive relationships are not necessarily causal.
- Never invent account names, scores, metrics or account-specific reasons.
- If account-level explanations are unavailable, say that directly.
- Give concise and actionable answers.
              `.trim(),
            },
            {
              role: 'system',
              content: `ACCOUNT SCORING DATA:\n${JSON.stringify(context)}`,
            },
            ...history
              .filter(
                (message) =>
                  message &&
                  ['user', 'assistant'].includes(message.role) &&
                  typeof message.content === 'string',
              )
              .map((message) => ({
                role: message.role,
                content: message.content.slice(0, 2000),
              })),
            {
              role: 'user',
              content: question,
            },
          ],
        }),
      },
    )

    const payload = await groqResponse.json()

    if (!groqResponse.ok) {
      throw new Error(
        payload?.error?.message || 'Groq could not answer the question.',
      )
    }

    const answer = payload?.choices?.[0]?.message?.content

    if (!answer) {
      throw new Error('The model returned an empty response.')
    }

    response.json({ answer })
  } catch (error) {
    console.error('Account Copilot error:', error)

    response.status(500).json({
      error: 'Account Copilot could not answer.',
      details:
        error instanceof Error ? error.message : 'Unknown server error',
    })
  }
})


function renderAccountCopilot(): string {
  return `
    <section class="usage-section ml-copilot">
      <div class="section-heading">
        <div>
          <p class="eyebrow">AI assistant</p>
          <h2>Ask the Account Scoring Copilot</h2>
        </div>
        <span>Grounded in current Snowflake model results</span>
      </div>

      <div class="ml-copilot-panel">
        <div id="account-copilot-messages" class="ml-copilot-messages">
          <div class="ml-copilot-message assistant">
            Ask which accounts to prioritize, compare models, or explain
            model performance.
          </div>
        </div>

        <div class="ml-copilot-suggestions">
          <button type="button" data-copilot-question=
            "Which five accounts should sales prioritize, and why?">
            Top five accounts
          </button>

          <button type="button" data-copilot-question=
            "Why was the deployed model selected?">
            Explain model selection
          </button>

          <button type="button" data-copilot-question=
            "What are the strongest global model drivers?">
            Explain model drivers
          </button>
        </div>

        <form id="account-copilot-form" class="ml-copilot-form">
          <textarea
            id="account-copilot-input"
            maxlength="1000"
            rows="3"
            placeholder="Ask about accounts, scores, models, or drivers..."
            required
          ></textarea>

          <button type="submit" class="primary-action">
            Ask Copilot
          </button>
        </form>
      </div>
    </section>
  `
}